# Spark vs Flink Fault Tolerance — Comparative Analysis
**Author:** Rajshekar Medipally | **Repo:** github.com/rmedipallycic/spark-streaming-fault-tolerance

720 trials (360 Spark + 360 Flink) across 6 strategies × 4 failure scenarios.

| System | Strategy | Mechanism | Interval |
|--------|----------|-----------|----------|
| Spark | A | High-frequency micro-batch checkpoint | 1s |
| Spark | B | Interval-based checkpoint | 30s |
| Spark | C | Async WAL checkpoint | 10s |
| Flink | F1 | Aligned Chandy-Lamport barriers | 10s |
| Flink | F2 | Unaligned barrier snapshots | 10s |
| Flink | F3 | Incremental RocksDB snapshots | 30s |


In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Load data
spark_df = pd.read_csv('../experiments/summary.csv')
flink_df = pd.read_csv('../experiments/flink/flink_summary.csv')
print(f'Spark: {len(spark_df)} configs | Flink: {len(flink_df)} configs')


## 1. Baseline Throughput

In [2]:
colors = ['#2196F3','#FF5722','#4CAF50','#9C27B0','#FF9800','#00BCD4']
labels = ['Spark A\n1s','Spark B\n30s','Spark C\nWAL','Flink F1\nAligned','Flink F2\nUnaligned','Flink F3\nIncremental']
tp  = [41165,51268,46743,54800,58200,52100]
tp_std = [1784,2488,1538,2100,2400,1900]

fig, ax = plt.subplots(figsize=(9,4))
x = np.arange(6)
bars = ax.bar(x, tp, yerr=tp_std, color=colors, alpha=0.85, capsize=4, edgecolor='white', width=0.6)
ax.axvline(2.5, color='gray', linestyle='--', alpha=0.4)
ax.text(1.0,62000,'Spark',ha='center',fontsize=9,fontweight='bold')
ax.text(4.0,62000,'Flink',ha='center',fontsize=9,fontweight='bold')
ax.set_title('Baseline Throughput: Spark vs Flink (30 trials × 1M records)',fontsize=10,fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(labels,fontsize=8)
ax.set_ylim(0,66000)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_:f'{v/1000:.0f}k'))
ax.grid(axis='y',alpha=0.3); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
for bar,val in zip(bars,tp):
    ax.text(bar.get_x()+bar.get_width()/2,val+600,f'{val/1000:.1f}k',ha='center',fontsize=8,fontweight='bold')
plt.tight_layout(); plt.show()


Flink F2 achieves **58,200 rec/s** (highest). Spark A is lowest at **41,165 rec/s**. Flink F1 vs Spark B gap is only **6.9%** — Spark's correctness risk is not offset by proportional throughput gains.

## 2. Recovery Latency — Driver/JobManager Failure

In [3]:
lat = [5244,20808,8878,7560,10440,13110]

fig, ax = plt.subplots(figsize=(9,4))
bars = ax.bar(x, lat, color=colors, alpha=0.85, edgecolor='white', width=0.6)
ax.axvline(2.5, color='gray', linestyle='--', alpha=0.4)
ax.text(1.0,22500,'Spark',ha='center',fontsize=9,fontweight='bold')
ax.text(4.0,22500,'Flink',ha='center',fontsize=9,fontweight='bold')
ax.set_title('Recovery Latency — Driver/JobManager Failure',fontsize=10,fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(labels,fontsize=8)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v,_:f'{v/1000:.1f}s'))
ax.grid(axis='y',alpha=0.3); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
for bar,val in zip(bars,lat):
    ax.text(bar.get_x()+bar.get_width()/2,val+200,f'{val/1000:.2f}s',ha='center',fontsize=8,fontweight='bold')
plt.tight_layout(); plt.show()


Spark B is the worst performer at **20,808ms**. Flink F1 delivers competitive recovery **(7,560ms)** with stronger correctness guarantees.

## 3. Silent Duplicate Rate — The Critical Finding

In [4]:
dup = [3.31,12.21,1.96,0.0,0.0,0.0]

fig, ax = plt.subplots(figsize=(9,4))
bars = ax.bar(x, dup, color=colors, alpha=0.85, edgecolor='white', width=0.6)
ax.axvline(2.5, color='gray', linestyle='--', alpha=0.4)
ax.text(1.0,14,'Spark',ha='center',fontsize=9,fontweight='bold')
ax.text(4.0,14,'Flink',ha='center',fontsize=9,fontweight='bold')
ax.set_title('Silent Duplicate Rate Under Checkpoint Corruption (no exception raised)',fontsize=10,fontweight='bold')
ax.set_xticks(x); ax.set_xticklabels(labels,fontsize=8)
ax.set_ylim(0,15.5); ax.set_ylabel('%')
ax.grid(axis='y',alpha=0.3); ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
for bar,val in zip(bars,dup):
    label = f'{val:.2f}%' if val>0 else '0.00%'
    ax.text(bar.get_x()+bar.get_width()/2,max(val,0)+0.2,label,ha='center',fontsize=8,fontweight='bold')
plt.tight_layout(); plt.show()


**Critical finding:** Spark produced silent duplicates at **1.96%–12.21%** with no exception raised. Flink produced **0.00%** across all 360 Flink trials. The Chandy-Lamport barrier protocol prevents silent duplicates by construction.

## 4. Throughput vs Correctness Trade-off

In [5]:
fig, ax = plt.subplots(figsize=(8,5))
systems = list(zip(labels,tp,dup,lat,colors))
for name,t,d,l,c in systems:
    size = max(80,(l/800)**1.5*200)
    ax.scatter(t/1000,d,s=size,color=c,alpha=0.8,edgecolors='white',linewidth=1,zorder=3)
    ax.annotate(name.replace('\n',' '),(t/1000,d),textcoords='offset points',
                xytext=(5,5 if d>0 else -12),fontsize=8,color=c,fontweight='bold')
ax.set_xlabel('Throughput (k rec/s)',fontsize=9)
ax.set_ylabel('Duplicate Rate Under Corruption (%)',fontsize=9)
ax.set_title('Throughput vs Correctness Trade-off\n(bubble size = recovery latency)',fontsize=10,fontweight='bold')
ax.grid(alpha=0.3); ax.set_ylim(-1,14); ax.set_xlim(36,63)
ax.spines['top'].set_visible(False); ax.spines['right'].set_visible(False)
plt.tight_layout(); plt.show()


All Flink strategies cluster at 0% duplicate rate with higher throughput. Spark B has the worst trade-off — moderate throughput + highest duplicate risk + slowest recovery.

## 5. Summary

In [6]:
print('System  Strategy  Throughput    Recovery(driver)  Dup Rate(corrupt)')
print('------  --------  ------------  ----------------  -----------------')
rows = [
    ('Spark','A','41,165 rec/s','5,244ms','3.31%',''),
    ('Spark','B','51,268 rec/s','20,808ms','12.21%','← worst'),
    ('Spark','C','46,743 rec/s','8,878ms','1.96%',''),
    ('Flink','F1','54,800 rec/s','7,560ms','0.00%',''),
    ('Flink','F2','58,200 rec/s','10,440ms','0.00%','← best throughput'),
    ('Flink','F3','52,100 rec/s','13,110ms','0.00%',''),
]
for r in rows:
    print(f'{r[0]:<7} {r[1]:<9} {r[2]:<13} {r[3]:<17} {r[4]:<8} {r[5]}')


## Conclusions

1. **Flink's barrier protocol is categorically safer** — 0% duplicate rate under corruption vs 1.96–12.21% for Spark.
2. **Throughput gap is smaller than expected** — Flink F1 vs Spark B is only 6.9%.
3. **Spark C (WAL) is the best Spark option** — but still 1.96% vs Flink's 0.00%.
4. **Recommendation for ML pipelines: Flink F1** — best correctness + competitive throughput + moderate recovery.

### Open Questions
- How do failure modes compose across multi-stage pipelines?
- What is the measurable downstream ML model accuracy impact of Spark B's 12.21% duplicate rate?
- Can hybrid Spark + Flink architectures achieve better trade-offs?

*Raw data: `experiments/raw/` (Spark) and `experiments/flink/raw/` (Flink).*